In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/cleaned/telecom_churn_cleaned.csv")

df.head()

,customer_id,telecom_partner,gender,age,state,city,pincode,date_of_registration,num_dependents,estimated_salary,calls_made,sms_sent,data_used,churn
0,1,Reliance Jio,F,25,Karnataka,Kolkata,755597,2020-01-01,4,124962,44.0,45.0,NaN,0
1,2,Reliance Jio,F,55,Mizoram,Mumbai,125926,2020-01-01,2,130556,62.0,39.0,5973.0,0
2,3,Vodafone,F,57,Arunachal Pradesh,Delhi,423976,2020-01-01,0,148828,49.0,24.0,193.0,1
3,4,BSNL,M,46,Tamil Nadu,Kolkata,522841,2020-01-01,1,38722,80.0,25.0,9377.0,1
4,5,BSNL,F,26,Tripura,Delhi,740247,2020-01-01,2,55098,78.0,15.0,1393.0,0


In [2]:
df.shape

(243553, 14)

In [3]:
df["churn"].value_counts()

churn
0    194726
1     48827
Name: count, dtype: int64

In [4]:
df["date_of_registration"] = pd.to_datetime(
    df["date_of_registration"],
    errors="coerce"
)

analysis_date = df["date_of_registration"].max()

df["tenure_days"] = (
    analysis_date - df["date_of_registration"]
).dt.days

df["tenure_months"] = (
    df["tenure_days"] / 30.44
).round(1)

df["registration_year"] = (
    df["date_of_registration"].dt.year
)

In [5]:
features = [
    "telecom_partner",
    "gender",
    "age",
    "state",
    "city",
    "num_dependents",
    "estimated_salary",
    "calls_made",
    "sms_sent",
    "data_used",
    "tenure_months",
    "registration_year"
]

target = "churn"

X = df[features]
y = df[target]

print("Features:", X.shape)
print("Target:", y.shape)

Features: (243553, 12)
Target: (243553,)


In [6]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (194842, 12)
X_test: (48711, 12)
y_train: (194842,)
y_test: (48711,)


In [7]:
random_state=42

In [8]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

In [9]:
numerical_features = [
    "age",
    "num_dependents",
    "estimated_salary",
    "calls_made",
    "sms_sent",
    "data_used",
    "tenure_months",
    "registration_year"
]

categorical_features = [
    "telecom_partner",
    "gender",
    "state",
    "city"
]

In [10]:
numerical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

In [11]:
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

In [12]:
preprocessor = ColumnTransformer([
    ("num", numerical_pipeline, numerical_features),
    ("cat", categorical_pipeline, categorical_features)
])

In [13]:
from sklearn.linear_model import LogisticRegression

logistic_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

In [14]:
logistic_model.fit(X_train, y_train)

,steps,"[('preprocessor', ...), ('classifier', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [15]:
logistic_model.fit(X_train, y_train)

print("Model training completed.")

Model training completed.


In [16]:
y_pred = logistic_model.predict(X_test)

print("Predictions generated:", len(y_pred))

Predictions generated: 48711


In [17]:
print(pd.Series(y_pred).value_counts())

0    48711
Name: count, dtype: int64


In [18]:
pd.Series(y_pred).value_counts()

0    48711
Name: count, dtype: int64

In [19]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report
)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)

y_prob = logistic_model.predict_proba(X_test)[:, 1]
roc_auc = roc_auc_score(y_test, y_prob)

print("Accuracy :", round(accuracy, 4))
print("Precision:", round(precision, 4))
print("Recall   :", round(recall, 4))
print("F1 Score :", round(f1, 4))
print("ROC-AUC  :", round(roc_auc, 4))

print("\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))

Accuracy : 0.7995
Precision: 0.0
Recall   : 0.0
F1 Score : 0.0
ROC-AUC  : 0.5059

Classification Report:
              precision    recall  f1-score   support

           0       0.80      1.00      0.89     38946
           1       0.00      0.00      0.00      9765

    accuracy                           0.80     48711
   macro avg       0.40      0.50      0.44     48711
weighted avg       0.64      0.80      0.71     48711



In [20]:
balanced_logistic_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42
    ))
])

In [21]:
balanced_logistic_model.fit(X_train, y_train)

print("Balanced model training completed.")

Balanced model training completed.


In [22]:
balanced_y_pred = balanced_logistic_model.predict(X_test)

print("Predictions generated:", len(balanced_y_pred))

Predictions generated: 48711


In [23]:
print(pd.Series(balanced_y_pred).value_counts())

0    24631
1    24080
Name: count, dtype: int64


In [24]:
balanced_accuracy = accuracy_score(y_test, balanced_y_pred)
balanced_precision = precision_score(
    y_test,
    balanced_y_pred,
    zero_division=0
)
balanced_recall = recall_score(
    y_test,
    balanced_y_pred,
    zero_division=0
)
balanced_f1 = f1_score(
    y_test,
    balanced_y_pred,
    zero_division=0
)

balanced_y_prob = balanced_logistic_model.predict_proba(X_test)[:, 1]

balanced_roc_auc = roc_auc_score(
    y_test,
    balanced_y_prob
)

print("Accuracy :", round(balanced_accuracy, 4))
print("Precision:", round(balanced_precision, 4))
print("Recall   :", round(balanced_recall, 4))
print("F1 Score :", round(balanced_f1, 4))
print("ROC-AUC  :", round(balanced_roc_auc, 4))

Accuracy : 0.5042
Precision: 0.2013
Recall   : 0.4964
F1 Score : 0.2864
ROC-AUC  : 0.5058


In [25]:
model_comparison = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Balanced Logistic Regression"
    ],
    "Accuracy": [
        accuracy,
        balanced_accuracy
    ],
    "Precision": [
        precision,
        balanced_precision
    ],
    "Recall": [
        recall,
        balanced_recall
    ],
    "F1 Score": [
        f1,
        balanced_f1
    ],
    "ROC-AUC": [
        roc_auc,
        balanced_roc_auc
    ]
})

model_comparison.round(4)

,Model,Accuracy,Precision,Recall,F1 Score,ROC-AUC
0,Logistic Regression,0.7995,0.0000,0.0000,0.0000,0.5059
1,Balanced Logistic Regression,0.5042,0.2013,0.4964,0.2864,0.5058


In [26]:
import joblib

In [27]:
joblib.dump(
    balanced_logistic_model,
    "../models/churn_model.joblib"
)

print("Model saved successfully.")

Model saved successfully.
